# 01 — Data Exploration

Load and inspect each data source, check data quality, preview schemas.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import json
from pathlib import Path
from uk_energy.config import *

## 1. BMRS — BM Units

In [ ]:
from uk_energy.ingest.bmrs import load_bm_units
bmu_df = load_bm_units()
print(f'Shape: {bmu_df.shape}')
bmu_df.head(10)

In [ ]:
# Check columns
print('Columns:', list(bmu_df.columns))
# Fuel type distribution if available
if 'fuelType' in bmu_df.columns:
    print('\nFuel type distribution:')
    print(bmu_df['fuelType'].value_counts().head(20))

## 2. WRI — Global Power Plant Database (GB subset)

In [ ]:
wri_path = PROCESSED_DIR / 'wri_gb_plants.csv'
if wri_path.exists():
    wri_df = pd.read_csv(wri_path)
    print(f'Shape: {wri_df.shape}')
    display(wri_df.head(10))
    print('\nPrimary fuel distribution:')
    print(wri_df['primary_fuel'].value_counts())
else:
    print('WRI data not yet downloaded — run: python -m uk_energy ingest --source wri')

## 3. REPD — Renewable Energy Planning Database

In [ ]:
repd_path = PROCESSED_DIR / 'repd_processed.csv'
if repd_path.exists():
    repd_df = pd.read_csv(repd_path, low_memory=False)
    print(f'Shape: {repd_df.shape}')
    print('\nStatus distribution:')
    print(repd_df['status'].value_counts() if 'status' in repd_df.columns else 'No status column')
    print('\nTechnology distribution (top 20):')
    if 'technology' in repd_df.columns:
        print(repd_df['technology'].value_counts().head(20))
    display(repd_df.head(5))
else:
    print('REPD not yet downloaded — run: python -m uk_energy ingest --source repd')

## 4. Carbon Intensity — Regional Generation Mix

In [ ]:
from uk_energy.ingest.carbon_intensity import parse_regional, parse_generation_mix

regional_df = parse_regional()
print('Regional data shape:', regional_df.shape)
display(regional_df.head(20))

In [ ]:
import plotly.express as px

if not regional_df.empty:
    # Pivot to get fuel breakdown per region
    pivot = regional_df.pivot_table(index='shortname', columns='fuel', values='perc', fill_value=0)
    print('Generation mix by region:')
    display(pivot.round(1))

## 5. Interconnectors Reference

In [ ]:
with open(INTERCONNECTORS_REF) as f:
    ic_data = json.load(f)

ic_df = pd.DataFrame(ic_data['interconnectors'])
print(f'Total interconnectors: {len(ic_df)}')
print(f'Total import capacity: {ic_df["capacity_mw"].sum():,} MW')
display(ic_df[['id', 'name', 'capacity_mw', 'countries', 'length_km', 'commissioned_year']].sort_values('capacity_mw', ascending=False))

## 6. OSUKED Cross-Reference

In [ ]:
from uk_energy.ingest.osuked import load_dictionary, load_plant_locations

osuked_dict = load_dictionary()
print(f'OSUKED dictionary: {len(osuked_dict)} entries')
print('Columns:', list(osuked_dict.columns))
display(osuked_dict.head(10))

## 7. Data Quality Summary

In [ ]:
summary = []
for name, path, key_col in [
    ('BM Units', BMRS_RAW / 'bm_units_all.json', None),
    ('WRI GB', PROCESSED_DIR / 'wri_gb_plants.csv', 'primary_fuel'),
    ('REPD', PROCESSED_DIR / 'repd_processed.csv', 'status'),
    ('DUKES', PROCESSED_DIR / 'dukes_processed.csv', 'fuel_type'),
]:
    if path.exists():
        size = path.stat().st_size / 1024
        summary.append({'source': name, 'file': path.name, 'size_kb': f'{size:.0f}', 'status': '✅'})
    else:
        summary.append({'source': name, 'file': path.name, 'size_kb': '-', 'status': '❌ Missing'})

display(pd.DataFrame(summary))